In [ ]:
import pandas as pd

In [ ]:
df = pd.read_parquet('modelling_dataset.parquet')

In [ ]:
df.head()

# Approach

In this cell, I am employing a 2 Step Approach Using XGBoost

### Step 1: XGB Classifier

Train an XGBoost Classifier on a subsampled training set (1:5 accident to non-accident ratio) 
to predict a binary target: $0$ (No Accident) vs. $1$ (At least one accident)

### Step 2: XGB Regressor

Filter dataset to only include rows where Accident_Count > 0. Train an XGBoost Regressor 
on this subset to predict the expected severity. Target is log-transformed ($\log(1 + \text{count})$)  to handle extreme outliers (counts up to 105), then converted back

### Step 3: Adding Models Together

Model A gives $P(\text{Accident})$. Model B gives $E[\text{Count} \mid \text{Accident}]$. 
Multiply them together to get the final Expected Risk Score:

$$\text{Risk Score} = P(\text{Accident}) \times E[\text{Count} \mid \text{Accident}]$$

## STEP 1: XBG Classifier

In [ ]:
# Creating a binary target variable for accident occurrence to use as target
df["Is_Accident"] = (df["Accident_Count"] > 0).astype(int)

In [ ]:
# Looking at where to split the time series data for training, validation, and testing
split_70 = df['Time_Stamp'].quantile(0.70)
split_85 = df['Time_Stamp'].quantile(0.85)

print(f"70% split: {split_70}")
print(f"85% split: {split_85}")

In [ ]:
# Creating the datasets for training, validation, and testing based on the time splits
train_df = df[df['Time_Stamp'] < split_70].copy()
val_df = df[(df['Time_Stamp'] >= split_70) & (df['Time_Stamp'] < split_85)].copy()
test_df = df[df['Time_Stamp'] >= split_85].copy()

# Verifying the sizes of the datasets
print(f"Training set size: {len(train_df)}")
print(f"Validation set size: {len(val_df)}")
print(f"Testing set size: {len(test_df)}")

In [ ]:
import numpy as np

def engineer_features(df, zone_stats=None):
    df_eng = df.copy()
    
    # interaction features
    df_eng['Wind_x_Precip'] = df_eng['Wind_Speed(mph)'] * df_eng['Precipitation(in)']
    df_eng['Hour_x_Weekend'] = df_eng['Hour'] * df_eng['Weekend']
    df_eng['BadWeather'] = (df_eng['Weather_Rain/Drizzle'] + df_eng['Weather_Stormy'] + df_eng['Weather_Visibility Issues']).clip(0, 1)
    df_eng['BadWeather_x_Hour'] = df_eng['BadWeather'] * df_eng['Hour']
    
    # non-linear relationships
    df_eng['Hour_squared'] = df_eng['Hour'] ** 2
    df_eng['WindSpeed_squared'] = df_eng['Wind_Speed(mph)'] ** 2
    df_eng['Precip_squared'] = df_eng['Precipitation(in)'] ** 2
    
    # cyclic encoding that tells model that hour 23 and hour 0 are close together
    df_eng['Hour_sin'] = np.sin(2 * np.pi * df_eng['Hour'] / 24)
    df_eng['Hour_cos'] = np.cos(2 * np.pi * df_eng['Hour'] / 24)
    df_eng['Month_sin'] = np.sin(2 * np.pi * df_eng['Month'] / 12)
    df_eng['Month_cos'] = np.cos(2 * np.pi * df_eng['Month'] / 12)
    df_eng['DayOfWeek_sin'] = np.sin(2 * np.pi * df_eng['Day_of_Week'] / 7)
    df_eng['DayOfWeek_cos'] = np.cos(2 * np.pi * df_eng['Day_of_Week'] / 7)
    
    # zone stats: compute from training only, reuse for val/test
    if zone_stats is None:
        # only called when processing training data
        zone_stats = (df.groupby('Zone_Int_ID')['Accident_Count']
                        .agg(['mean', 'std', 'max'])
                        .reset_index())
        zone_stats.columns = ['Zone_Int_ID', 'Zone_Mean', 'Zone_Std', 'Zone_Max']
        zone_stats['Zone_Std'] = zone_stats['Zone_Std'].fillna(0)
    
    df_eng = df_eng.merge(zone_stats, on='Zone_Int_ID', how='left')
    
    return df_eng, zone_stats

In [ ]:
# apply to training data first, this computes zone_stats from training only
train_df_eng, zone_stats = engineer_features(train_df)

# pass zone_stats into val and test
val_df_eng,  _ = engineer_features(val_df,  zone_stats=zone_stats)
test_df_eng, _ = engineer_features(test_df, zone_stats=zone_stats)

print("New columns added:", [c for c in train_df_eng.columns if c not in train_df.columns])

In [ ]:
new_features = [
    'Wind_x_Precip', 'Hour_x_Weekend', 'BadWeather', 'BadWeather_x_Hour', 'Hour_squared', 'WindSpeed_squared', 'Precip_squared',
    'Hour_sin', 'Hour_cos', 'Month_sin', 'Month_cos', 'DayOfWeek_sin', 'DayOfWeek_cos', 'Zone_Mean', 'Zone_Std', 'Zone_Max']

# base features
features = [
    'Hour', 'Day_of_Week', 'Month', 'Weekend', 'Holiday', 'Year', 'Zone_Int_ID', 'Amenity', 'Crossing', 'Give_Way', 'Junction',
    'Railway', 'Station', 'Stop', 'Traffic_Signal', 'City_Houston', 'City_Miami', 'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Clear', 'Weather_Cloudy', 'Weather_Dust/Windy',
    'Weather_Rain/Drizzle', 'Weather_Snow/Ice', 'Weather_Stormy', 'Weather_Visibility Issues'
]

# add engineered features on top
regressor_features = features + new_features

print(f"Total regressor features: {len(regressor_features)}")

In [ ]:
# subsampling the training set to provide more balance to the classes for the classification model (not needed for regression)
accidents = train_df_eng[train_df_eng['Is_Accident'] == 1]
non_accidents = train_df_eng[train_df_eng['Is_Accident'] == 0]

non_accidents_sampled = non_accidents.sample(n=len(accidents) * 5, random_state=42)

train_subsampled = pd.concat([accidents, non_accidents_sampled]).sample(frac=1, random_state=42)

print(f"Subsampled training set size: {len(train_subsampled)}")

In [ ]:
# Zone_Mean, Zone_Std, Zone_Max ARE zone stats from Accident_Count
# but they're safe because engineer_features computed them from training data only
classifier_features = regressor_features

# created X and y for the training, validation, and test sets for classification model
X_train_classifier = train_subsampled[classifier_features]
y_train_classifier = train_subsampled['Is_Accident']

X_val_classifier = val_df_eng[classifier_features]
y_val_classifier = val_df_eng['Is_Accident']

X_test_classifier = test_df_eng[classifier_features]
y_test_classifier = test_df_eng['Is_Accident']

In [ ]:
from xgboost import XGBClassifier

# Training the model on the subsetted training data
classifier_model = XGBClassifier(
    n_estimators=150, # number of trees to build
    learning_rate=0.05, # how much the model updates prediction
    max_depth=5, #max tree depth
    scale_pos_weight=5,  # model gives 5x more importance to unbalanced class
    objective='binary:logistic', # binary classification
    eval_metric='logloss', # log loss as error
    subsample=0.8, # model randomly samples 80% of train data for each tree
    colsample_bytree=0.8, # model randomly samples 80% of cols fir each tree
    random_state=42 
)

classifier_model.fit(X_train_classifier, y_train_classifier)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, average_precision_score
import matplotlib.pyplot as plt

# Evaluating the initial model on the validation set 
y_val_preds = classifier_model.predict(X_val_classifier)
y_val_pred_proba = classifier_model.predict_proba(X_val_classifier)[:, 1]

# Checking metrics
print("Classification Report:")
print(classification_report(y_val_classifier, y_val_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_val_classifier, y_val_preds))

print(f"ROC AUC Score: {roc_auc_score(y_val_classifier, y_val_pred_proba):.4f}")

print(f"Average Precision Score: {average_precision_score(y_val_classifier, y_val_pred_proba)}")


In [ ]:
import numpy as np
from sklearn.model_selection import GridSearchCV, PredefinedSplit

X_tuning_classifier = pd.concat([X_train_classifier, X_val_classifier])
y_tuning_classifier = pd.concat([y_train_classifier, y_val_classifier])

# Identifying which is used for training (-1) and which is used for validation(0)
split_index = np.full(X_tuning_classifier.shape[0], -1)
split_index[len(train_subsampled):] = 0
pds = PredefinedSplit(test_fold=split_index)

# Reduced parameter grid for faster tuning (16 combinations instead of 72)
param_grid = {
    'max_depth': [5, 7],
    'learning_rate': [0.05, 0.1],
    'n_estimators': [100, 150],
    'scale_pos_weight': [5, 15],
}

grid_search = GridSearchCV(
    estimator=XGBClassifier(objective='binary:logistic', eval_metric='logloss', random_state=42),
    param_grid=param_grid,
    cv=pds,
    scoring="average_precision",  
    verbose=3,
    n_jobs=4
)

grid_search.fit(X_tuning_classifier, y_tuning_classifier)

best_classifier = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")

In [ ]:
# Determining Optimal Threshold for Classification
from sklearn.metrics import precision_recall_curve
import matplotlib.pyplot as plt

# Using the best classifier on the validation set
y_val_probs = best_classifier.predict_proba(val_df_eng[regressor_features])[:, 1]
precision, recall, thresholds = precision_recall_curve(val_df_eng['Is_Accident'], y_val_probs)

# Plotting precision recall curve for visual analysis
plt.figure(figsize=(8, 4))
plt.plot(thresholds, precision[:-1], label="Precision")
plt.plot(thresholds, recall[:-1], label="Recall")
plt.xlabel("Threshold")
plt.legend()
plt.grid(True)
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Testing final model on test dataset
y_test_pred_proba = best_classifier.predict_proba(X_test_classifier)[:, 1]

# Use a 0.3 threshold based on above precision-recall curve analysis 
y_test_preds = (y_test_pred_proba >= 0.4).astype(int)

# Checking metrics
print("Classification Report:")
print(classification_report(y_test_classifier, y_test_preds))

print("Confusion Matrix:")
print(confusion_matrix(y_test_classifier, y_test_preds))

print(f"ROC AUC Score: {roc_auc_score(y_test_classifier, y_test_pred_proba):.4f}")

print(f"Average Precision Score: {average_precision_score(y_test_classifier, y_test_pred_proba)}")


## STEP 2: XGB Regressor

In [ ]:
# look at only accidents to train regression model (filter accident rows)
train_accidents = train_df_eng[train_df_eng['Is_Accident'] == 1].copy()
val_accidents = val_df_eng[val_df_eng['Is_Accident'] == 1].copy()

X_train_regressor = train_accidents[regressor_features]
y_train_regressor = train_accidents['Accident_Count'] # Use accident count as target for regression model

X_val_regressor = val_accidents[regressor_features]
y_val_regressor = val_accidents['Accident_Count']

# use log tranformations to remove large outliers
y_train_log = np.log1p(y_train_regressor) 
y_val_log = np.log1p(y_val_regressor)

# verification of regression datasets
print(f"Training on {len(X_train_regressor)} accident rows")
print(f"Validating on {len(X_val_regressor)} accident rows")

In [ ]:
from xgboost import XGBRegressor
from sklearn.model_selection import GridSearchCV, PredefinedSplit

X_tuning_regressor = pd.concat([X_train_regressor, X_val_regressor]).reset_index(drop=True)
y_tuning_log = pd.concat([y_train_log, y_val_log]).reset_index(drop=True)

split_index = np.full(len(X_tuning_regressor), -1)
split_index[len(X_train_regressor):] = 0
pds = PredefinedSplit(test_fold=split_index)

# through trial and error this was the best set of parameters found for the regression model
param_grid_regressor = {
    'max_depth': [6],
    'learning_rate': [0.1],
    'n_estimators': [500],
    'subsample': [0.8],
    'colsample_bytree': [0.7],
    'min_child_weight': [1],
}

grid_search_regressor = GridSearchCV(
    estimator=XGBRegressor(objective='reg:squarederror', random_state=42),
    param_grid=param_grid_regressor,
    cv=pds,
    scoring='neg_mean_absolute_error',
    verbose=1,
    n_jobs=-1
)

grid_search_regressor.fit(X_tuning_regressor, y_tuning_log)
best_regressor = grid_search_regressor.best_estimator_
print(f"Best Parameters: {grid_search_regressor.best_params_}")

In [ ]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

y_val_log_preds   = best_regressor.predict(val_accidents[regressor_features])
y_val_final_preds = np.expm1(y_val_log_preds)

mae = mean_absolute_error(y_val_regressor, y_val_final_preds)
rmse = root_mean_squared_error(y_val_regressor, y_val_final_preds)
r2_raw = r2_score(y_val_regressor, y_val_final_preds)
r2_log = r2_score(y_val_log, y_val_log_preds)  # what model actually optimized

print(f"MAE: {round(mae, 4)}")
print(f"RMSE: {round(rmse, 4)}")
print(f"R² (raw): {round(r2_raw, 4)}")
print(f"R² (log space): {round(r2_log, 4)}")

## STEP 3: Final Synthesis

In [ ]:
# # Using the classifier to predict the probability of accident on test
# test_probs = best_classifier.predict_proba(X_test_classifier)[:, 1]
# test_preds = (test_probs >= 0.4).astype(int)

# # Using the regressor to predict the number of accidents on test
# test_log_severity = best_regressor.predict(X_test_classifier)
# test_severity = np.expm1(test_log_severity)


# final_results = test_df.copy() 
# final_results['Prob_Accident'] = test_probs
# final_results['Predicted_Accident'] = test_preds
# final_results['Expected_Severity'] = test_severity

# # Creating the final risk score by multiplying the probability of an accident occurring with the expected number of accidents
# final_results['Risk_Score'] = test_probs * test_severity
# final_results["Risk_Score_2"] = test_preds * test_severity

In [ ]:
# using the classifier to predict the probability of accident on test
test_probs = best_classifier.predict_proba(X_test_classifier)[:, 1]
test_preds = (test_probs >= 0.4).astype(int)

# filter test set to only accident cases for regression
test_accidents = test_df_eng[test_df_eng['Is_Accident'] == 1].copy()
X_test_accidents = test_accidents[regressor_features]

# using the regressor to predict severity only on accident cases
test_log_severity_accidents = best_regressor.predict(X_test_accidents)
test_severity_accidents = np.expm1(test_log_severity_accidents)

# Initialize severity array with zeros for all test cases
test_severity = np.zeros(len(test_df_eng))
# Fill in severity predictions only for accident rows
accident_indices = test_df_eng[test_df_eng['Is_Accident'] == 1].index
test_severity[test_df_eng.index.isin(accident_indices)] = test_severity_accidents

final_results = test_df_eng.copy() 
final_results['Prob_Accident'] = test_probs
final_results['Predicted_Accident'] = test_preds
final_results['Expected_Severity'] = test_severity

# Creating the final risk score by multiplying the probability of an accident occurring with the expected number of accidents
final_results['Risk_Score'] = test_probs * test_severity
final_results["Risk_Score_2"] = test_preds * test_severity


In [ ]:
final_results[final_results['Accident_Count'] > 1]

In [ ]:
final_results.to_parquet('final_results.parquet', engine='fastparquet')

## STEP 4: Visualizing Results

#### Looking at the actual accidents in the test set

In [ ]:
import plotly.express as px
import h3

# Aggregate actual accident counts from test set by zone
miami_test = final_results[final_results['City_Miami'] == 1].copy()

zone_accidents = miami_test.groupby('Zone_ID')['Accident_Count'].sum().reset_index()
zone_accidents.rename(columns={'Accident_Count': 'Total_Accidents'}, inplace=True)

# Build GeoJSON from H3 zone IDs
geojson_features = []
for zone in zone_accidents['Zone_ID']:
    boundary = h3.cell_to_boundary(zone)
    coords = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])
    geojson_features.append({
        'type': 'Feature',
        'id': zone,
        'properties': {},
        'geometry': {'type': 'Polygon', 'coordinates': [coords]}
    })

geojson = {'type': 'FeatureCollection', 'features': geojson_features}

# Plot the map
fig = px.choropleth_map(
    zone_accidents,
    geojson=geojson,
    locations='Zone_ID',
    featureidkey='id',
    color='Total_Accidents',
    color_continuous_scale='Reds',
    map_style='carto-positron',
    zoom=10,
    #center={'lat': 29.7604, 'lon': -95.3698},
    center={'lat': 25.7617, 'lon': -80.1918}, #for Miami
    title='Test Set: Actual Accident Counts by Zone',
    labels={'Total_Accidents': 'Accidents'}
)

fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()

In [ ]:
import plotly.express as px
import h3

# Aggregate predicted risk scores from test set by zone
miami_test = final_results[final_results['City_Miami'] == 1].copy()

zone_risk = miami_test.groupby('Zone_ID')['Risk_Score'].sum().reset_index()
zone_risk.rename(columns={'Risk_Score': 'Total_Risk_Score'}, inplace=True)

# Build GeoJSON from H3 zone IDs
features = []
for zone in zone_risk['Zone_ID']:
    boundary = h3.cell_to_boundary(zone)
    coords = [[lon, lat] for lat, lon in boundary]
    coords.append(coords[0])
    features.append({
        'type': 'Feature',
        'id': zone,
        'properties': {},
        'geometry': {'type': 'Polygon', 'coordinates': [coords]}
    })

geojson = {'type': 'FeatureCollection', 'features': features}

# Plot the map
fig = px.choropleth_map(
    zone_risk,
    geojson=geojson,
    locations='Zone_ID',
    featureidkey='id',
    color='Total_Risk_Score',
    color_continuous_scale='Reds',
    map_style='carto-positron',
    zoom=10,
    #center={'lat': 29.7604, 'lon': -95.3698},
    center={'lat': 25.7617, 'lon': -80.1918}, #for Miami
    title='Test Set: Predicted Risk Score by Zone',
    labels={'Total_Risk_Score': 'Risk Score'}
)

fig.update_layout(margin={'r': 0, 't': 30, 'l': 0, 'b': 0})
fig.show()

In [ ]:
# Having the models saved for reproducibility and future use without needing to retrain

import joblib

joblib.dump(best_classifier, 'xgb_classifier_model.pkl')
joblib.dump(best_regressor, 'xgb_regressor_model.pkl')